# AINPC — '어둠(Darkness)' 인격 파인튜닝 (Colab)

**시작 전 필수:** 상단 메뉴 → `런타임` → `런타임 유형 변경` → 하드웨어 가속기 **T4 GPU** 선택.

**실행 순서:** 위에서 아래로 셀을 순서대로 실행하세요.
런타임이 끊겨도 결과가 사라지지 않도록 **2) 구글 드라이브 연결**을 권장합니다.
한 번 학습한 뒤 런타임이 끊겼다면, 1·2·3·5번만 실행하면 저장된 어댑터를 자동으로 불러와
**학습(7) 없이 바로 GGUF(9)** 로 넘어갈 수 있습니다.


## 1) GPU 확인 (T4가 보여야 정상)

In [ ]:
!nvidia-smi

## 2) (권장) 구글 드라이브 연결 — 런타임이 끊겨도 모델 보존
`USE_DRIVE = False` 로 두면 임시 런타임에만 저장되어 끊기면 사라집니다.

In [ ]:
import os
USE_DRIVE = True   # False = 임시 저장(끊기면 소실)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/AINPC'
else:
    SAVE_DIR = '/content/AINPC'

os.makedirs(SAVE_DIR, exist_ok=True)
ADAPTER_DIR = os.path.join(SAVE_DIR, 'darkness-lora')
GGUF_DIR    = os.path.join(SAVE_DIR, 'darkness-gguf')
print('SAVE_DIR =', SAVE_DIR)

## 3) 설치 (2~3분 소요)

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade trl datasets transformers

## 4) 학습 데이터 업로드
실행하면 파일 선택창이 뜹니다. PC의 `Training/data/train_data.jsonl` 을 선택하세요.
(이미 학습을 끝내고 GGUF만 다시 만들 거라면 이 셀은 건너뛰어도 됩니다.)

In [ ]:
from google.colab import files
import os
print('train_data.jsonl 파일을 선택하세요 ↓')
up = files.upload()
os.makedirs('data', exist_ok=True)
fn = list(up.keys())[0]
with open('data/train_data.jsonl', 'wb') as f:
    f.write(up[fn])
n = sum(1 for _ in open('data/train_data.jsonl', encoding='utf-8'))
print(f'저장됨 -> data/train_data.jsonl  (대화 {n}줄)')

## 5) 모델 로드 (신규 학습 / 이어하기 자동 판별)
`SAVE_DIR` 에 저장된 어댑터가 있으면 그것을 불러오고(이어하기),
없으면 베이스 모델에 새 LoRA 어댑터를 붙입니다(신규 학습).

In [ ]:
import os, torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME     = "unsloth/Meta-Llama-3.1-8B-Instruct"
MAX_SEQ_LENGTH = 2048
LORA_RANK      = 16

resume = os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json'))
source = ADAPTER_DIR if resume else MODEL_NAME
print('이어하기: 저장된 어댑터 로드' if resume else '신규 학습: 베이스 모델 로드', '->', source)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = source,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

if not resume:
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_alpha=LORA_RANK, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=42,
    )
print('모델 준비 완료 (resume =', resume, ')')

## 6) 데이터셋 구성

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def format_sample(s):
    return {"text": tokenizer.apply_chat_template(
        s["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(load_jsonl('data/train_data.jsonl')).map(format_sample)
print('학습 샘플:', len(dataset))
print('--- 예시 ---')
print(dataset[0]['text'][:600])

## 7) 학습 + 어댑터 저장 (T4 기준 8B·1000대화 약 20~40분)
학습이 끝나면 어댑터를 `SAVE_DIR` 에 자동 저장합니다.
**이미 학습한 어댑터를 불러온 경우(이어하기), 이 셀은 건너뛰고 8·9로 가세요.**

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

if 'model' not in globals():
    raise RuntimeError("model 이 없습니다. 먼저 5) 모델 로드 를 실행하세요.")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = 2048,
        dataset_num_proc            = 2,
        packing                     = False,
        padding_free                = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps     = 10,
        optim             = "adamw_8bit",
        weight_decay      = 0.01,
        lr_scheduler_type = "cosine",
        warmup_steps      = 10,
        output_dir        = "output/darkness-lora-ckpt",
        save_strategy     = "epoch",
        seed              = 42,
        report_to         = "none",
    ),
)
trainer.train()

# 어댑터 영구 저장 (런타임이 끊겨도 5번에서 자동 복구됨)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('학습 완료 + 어댑터 저장 ->', ADAPTER_DIR)

## 8) (선택) 빠른 테스트 — 어둠이 JSON으로 답하는지 확인

In [ ]:
if 'model' not in globals():
    raise RuntimeError("model 이 없습니다. 먼저 5) 모델 로드 를 실행하세요.")

FastLanguageModel.for_inference(model)
msgs = [
    {"role":"system","content":"너는 이 캐릭터의 또 다른 인격 '어둠'이야. 반드시 {\"dialogue\":\"\",\"mood_delta\":0} JSON으로만 답해. 현재 상황: HP 10/100, 기분 20/100, 주변 적 5명, 현재 플레이어가 몸을 제어 중."},
    {"role":"user","content":"대화를 시작해."},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=80, temperature=0.85, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 9) GGUF 변환 (Q4_K_M) — Unity 로 가져갈 파일
먼저 unsloth 내장 변환을 시도하고, 실패하면 llama.cpp 로 직접 변환(폴백)합니다.
시간이 다소 걸릴 수 있습니다.

In [ ]:
import os, glob, sys, subprocess

if 'model' not in globals():
    raise RuntimeError("model 이 없습니다. 먼저 5) 모델 로드 (필요시 7) 학습) 를 실행하세요.")

os.makedirs(GGUF_DIR, exist_ok=True)

def convert_with_unsloth():
    model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

def convert_with_llamacpp():
    # 1) 16bit 병합
    merged = os.path.join(SAVE_DIR, 'darkness-merged')
    model.save_pretrained_merged(merged, tokenizer, save_method="merged_16bit")
    # 2) llama.cpp 준비
    if not os.path.exists('llama.cpp'):
        subprocess.run(['git','clone','--depth=1',
                        'https://github.com/ggerganov/llama.cpp'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',
                    'llama.cpp/requirements.txt'], check=True)
    # 3) f16 GGUF 변환
    f16 = os.path.join(GGUF_DIR, 'darkness-f16.gguf')
    subprocess.run([sys.executable,'llama.cpp/convert_hf_to_gguf.py',
                    merged,'--outfile',f16,'--outtype','f16'], check=True)
    # 4) Q4_K_M 양자화 (quantize 바이너리 빌드)
    subprocess.run(['cmake','-B','llama.cpp/build','llama.cpp'], check=True)
    subprocess.run(['cmake','--build','llama.cpp/build','--config','Release',
                    '-j','--target','llama-quantize'], check=True)
    qbin = 'llama.cpp/build/bin/llama-quantize'
    q = os.path.join(GGUF_DIR, 'darkness-Q4_K_M.gguf')
    subprocess.run([qbin, f16, q, 'Q4_K_M'], check=True)
    if os.path.exists(f16):
        os.remove(f16)

try:
    print('unsloth 방식으로 GGUF 변환 시도...')
    convert_with_unsloth()
except Exception as e:
    print('unsloth 변환 실패 -> llama.cpp 직접 변환으로 폴백')
    print('사유:', repr(e))
    convert_with_llamacpp()

g = glob.glob(os.path.join(GGUF_DIR, '*.gguf'))[0]
print('\nGGUF 생성 완료:', g, round(os.path.getsize(g)/1e9, 2), 'GB')

## 10) GGUF 다운로드 → Unity LLM 컴포넌트 'Load model' 에 지정
드라이브를 연결했다면 `MyDrive/AINPC/darkness-gguf/` 에도 이미 저장돼 있습니다.

In [ ]:
from google.colab import files
import glob, os
files.download(glob.glob(os.path.join(GGUF_DIR, '*.gguf'))[0])